# Run `f7d2c378-bca2-46fe-b5a6-47062fb75140` Probe

This notebook mirrors the run configuration of `f7d2c378-bca2-46fe-b5a6-47062fb75140` using the same experimental style as `01_run_322_btcusdt_1h_artifact_probe.ipynb`.

Important differences from the five-indicator notebook:
- this run uses two indicators only: `ma.dema` and `ma.hma`
- the runtime path is no-risk by default (`best_tp_pct` / `best_sl_pct` are `NULL` in persisted results)
- the notebook still keeps the same universal patterns: row prefilter, combo proxy prefilter, trade-list-first exact scoring, and fast-vs-reference self-check


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import heapq
import itertools
import math

import numba as nb
import numpy as np
import yaml

RUN_ID = "f7d2c378-bca2-46fe-b5a6-47062fb75140"
TIMEFRAME = "15m"
RUN_TIME_RANGE_START = datetime(2020, 1, 11, 20, 8, tzinfo=timezone.utc)
RUN_TIME_RANGE_END = datetime(2026, 4, 11, 20, 8, tzinfo=timezone.utc)
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)

ARTIFACT_ROOT = Path("/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_b")
PRICE_DIR_15M = ARTIFACT_ROOT / "prices" / TIMEFRAME
PRICE_DIR_1M = ARTIFACT_ROOT / "prices" / "1m"
MAPPINGS_15M_DIR = ARTIFACT_ROOT / "mappings" / TIMEFRAME

COMMON_PRICE_SOURCES = ["close", "hlc3", "ohlc4", "low", "high", "open"]
RUN_INDICATORS_15M = {
    "ma.dema": {
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "ma.hma": {
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
}

REQUEST_INDICATOR_GRIDS = [
    {
        "indicator_id": "ma.dema",
        "sources": ["close", "high", "hlc3"],
        "window_range": [5, 200],
    },
    {
        "indicator_id": "ma.hma",
        "sources": ["close", "high", "hlc3"],
        "window_range": [5, 200],
    },
]

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_15M_PATH = PRICE_DIR_15M / "open_time.i64.npy"
PRICE_CLOSE_TIME_15M_PATH = PRICE_DIR_15M / "close_time.i64.npy"
PRICE_OHLCV_15M_PATH = PRICE_DIR_15M / "ohlcv.f32.npy"
PRICE_OPEN_TIME_1M_PATH = PRICE_DIR_1M / "open_time.i64.npy"
PRICE_CLOSE_TIME_1M_PATH = PRICE_DIR_1M / "close_time.i64.npy"
PRICE_OHLCV_1M_PATH = PRICE_DIR_1M / "ohlcv.f32.npy"
BAR_OPEN_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_open_1m_idx.u32.npy"
BAR_CLOSE_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_close_1m_idx.u32.npy"

SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "signals.i8.npy",
    }
    for indicator_id in RUN_INDICATORS_15M
}

USE_MMAP = True
ROW_PREFILTER_ROWS_PER_INDICATOR = 142
PREFILTER_TOP_FRAC = ROW_PREFILTER_ROWS_PER_INDICATOR / 588.0
PREFILTER_MIN_NONZERO = 1
COMBO_PREFILTER_TOP_FRAC = 1.0
COMBO_MIN_CONFIRM = 1
TIME_CHUNK = 4096
COMBO_CHUNK_SIZE = 4096
TOP_K_DEFAULT = 100
SELF_CHECK_N_DEFAULT = 8
FEE_RATE = 0.00075
SLIPPAGE_RATE = 0.0001
INIT_CASH_QUOTE = 10000.0
FIXED_QUOTE = 100.0
SAFE_PROFIT_PERCENT = 30.0
USE_FIXED_QUOTE = False
USE_PROFIT_LOCK = False
BARS_PER_YEAR_EXEC_1M = 365.0 * 24.0 * 60.0
CLOSE_ON_END = np.int8(1)
NEG_INF = np.float32(-1e30)
NEG_LARGE = -1.0e30

PATHS_TO_CHECK = [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_15M_PATH,
    PRICE_CLOSE_TIME_15M_PATH,
    PRICE_OHLCV_15M_PATH,
    PRICE_OPEN_TIME_1M_PATH,
    PRICE_CLOSE_TIME_1M_PATH,
    PRICE_OHLCV_1M_PATH,
    BAR_OPEN_1M_IDX_15M_PATH,
    BAR_CLOSE_1M_IDX_15M_PATH,
] + [value for item in SIGNAL_PATHS_15M.values() for value in item.values()]

for path in PATHS_TO_CHECK:
    print(f"{path}: exists={path.exists()}")


In [ ]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
price_manifest_15m = {item["timeframe"]: item for item in slot_manifest["prices"]}[TIMEFRAME]
signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in SIGNAL_PATHS_15M.items()
}

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("15m price bars:", price_manifest_15m["coverage"]["bar_count"])
for indicator_id, manifest in signal_manifests_15m.items():
    print(indicator_id, "rows_count=", manifest["rows_count"], "shape=", tuple(manifest["signals"]["shape"]))
print("request indicator grids:", REQUEST_INDICATOR_GRIDS)
print("row prefilter fraction:", PREFILTER_TOP_FRAC)


In [ ]:
price_open_time_15m = np.load(PRICE_OPEN_TIME_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_close_time_15m = np.load(PRICE_CLOSE_TIME_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_ohlcv_15m = np.load(PRICE_OHLCV_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_open_time_1m = np.load(PRICE_OPEN_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_close_time_1m = np.load(PRICE_CLOSE_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_ohlcv_1m = np.load(PRICE_OHLCV_1M_PATH, mmap_mode="r" if USE_MMAP else None)
bar_open_1m_idx_15m = np.load(BAR_OPEN_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
bar_close_1m_idx_15m = np.load(BAR_CLOSE_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
full_signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r" if USE_MMAP else None)
    for indicator_id, paths in SIGNAL_PATHS_15M.items()
}

price_fields_15m = {
    "open": np.asarray(price_ohlcv_15m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_15m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_15m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_15m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_15m[:, 4], dtype=np.float32),
}
price_fields_1m = {
    "open": np.asarray(price_ohlcv_1m[:, 0], dtype=np.float32),
    "close": np.asarray(price_ohlcv_1m[:, 3], dtype=np.float32),
}

time_mask_15m = (price_open_time_15m >= RUN_TIME_RANGE_START_MS) & (price_open_time_15m <= RUN_TIME_RANGE_END_MS)
time_mask_idx_15m = np.flatnonzero(time_mask_15m)
if time_mask_idx_15m.size == 0:
    raise ValueError("Run time range selects no 15m bars.")
time_slice_start_15m = int(time_mask_idx_15m[0])
time_slice_stop_15m = int(time_mask_idx_15m[-1]) + 1
USE_CONTIGUOUS_TIME_SLICE_15M = (time_slice_stop_15m - time_slice_start_15m) == int(time_mask_idx_15m.size)
time_selector_15m = slice(time_slice_start_15m, time_slice_stop_15m) if USE_CONTIGUOUS_TIME_SLICE_15M else time_mask_15m
run_bar_open_1m_idx_15m = np.asarray(bar_open_1m_idx_15m[time_selector_15m], dtype=np.int32)
run_bar_close_1m_idx_15m = np.asarray(bar_close_1m_idx_15m[time_selector_15m], dtype=np.int32)

signal_close_15m = np.asarray(price_fields_15m["close"][time_selector_15m], dtype=np.float32)
signal_returns_15m = np.ascontiguousarray(((signal_close_15m[1:] / signal_close_15m[:-1]) - 1.0).astype(np.float32))
n_signal_bars = int(signal_close_15m.shape[0])
n_signal_intervals = int(signal_returns_15m.shape[0])
T_exec_limit_1m = np.int32(int(run_bar_close_1m_idx_15m[-1]) + 1)
last_close_1m = float(price_fields_1m["close"][int(T_exec_limit_1m) - 1])

sig_entry_exec_idx_15m = np.empty(n_signal_bars, dtype=np.int32)
if n_signal_bars > 1:
    sig_entry_exec_idx_15m[:-1] = np.asarray(run_bar_open_1m_idx_15m[1:], dtype=np.int32)
sig_entry_exec_idx_15m[-1] = T_exec_limit_1m

print("signal bars 15m:", n_signal_bars)
print("signal intervals 15m:", n_signal_intervals)
print("execution limit 1m:", int(T_exec_limit_1m))
for indicator_id, matrix in full_signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)


In [ ]:
def njit_cached(*, parallel: bool = False, fastmath: bool = False, inline: str = "never"):
    """Return a notebook-friendly Numba decorator without filesystem cache dependence.

    Parameters:
        parallel: Whether to enable Numba parallel lowering.
        fastmath: Whether to enable relaxed floating-point optimizations.
        inline: Requested Numba inline policy.

    Returns:
        A decorator wrapping `numba.njit` with `cache=False`.

    Assumptions:
        The notebook executes in an environment where file-backed cache locators may be unavailable.

    Raises:
        None directly; Numba compilation errors propagate from the wrapped function.

    Side effects:
        Triggers Numba compilation on first call of the wrapped function.
    """
    def decorate(func):
        return nb.njit(parallel=parallel, fastmath=fastmath, inline=inline, cache=False)(func)
    return decorate


In [ ]:
def build_row_catalog(*, indicator_id, sources, param_name, param_values):
    """Build row metadata for one artifact-backed indicator matrix.

    Parameters:
        indicator_id: Indicator id matching the artifact directory name.
        sources: Ordered source axis for this artifact matrix.
        param_name: Human-readable parameter axis name.
        param_values: Ordered parameter values inside each source block.

    Returns:
        A list of dictionaries with `row_id`, `source`, and parameter value for every artifact row.

    Assumptions:
        The artifact layout is source-major and parameter-minor inside each source block.

    Raises:
        None.

    Side effects:
        None.
    """
    rows = []
    block = len(param_values)
    for source_idx, source_name in enumerate(sources):
        base = source_idx * block
        for offset, param_value in enumerate(param_values):
            rows.append({
                "indicator_id": indicator_id,
                "row_id": base + offset,
                "source": source_name,
                param_name: int(param_value),
            })
    return rows


def row_ids_for_sources(*, indicator_id, source_names):
    """Return original artifact row ids for the requested source blocks.

    Parameters:
        indicator_id: Indicator id present in `RUN_INDICATORS_15M`.
        source_names: Iterable of source names to select.

    Returns:
        An `int32` array with the original artifact row ids for the requested sources.

    Assumptions:
        Each source occupies one contiguous block of rows.

    Raises:
        KeyError: If the requested source is not present for the indicator.

    Side effects:
        None.
    """
    meta = RUN_INDICATORS_15M[indicator_id]
    param_count = len(meta["param_values"])
    source_to_index = {name: idx for idx, name in enumerate(meta["sources"])}
    row_ids = []
    for source_name in source_names:
        source_idx = source_to_index[source_name]
        start = source_idx * param_count
        row_ids.extend(range(start, start + param_count))
    return np.asarray(row_ids, dtype=np.int32)


def extract_signal_rows(*, indicator_id, row_ids):
    """Load a bounded set of 15m signal rows for one indicator into a contiguous matrix.

    Parameters:
        indicator_id: Indicator id present in `full_signal_matrices_15m`.
        row_ids: Original artifact row ids to extract.

    Returns:
        Int8 matrix with shape `(len(row_ids), n_signal_bars)` over the run time range.

    Assumptions:
        `time_mask_15m` already bounds the active experiment range.

    Raises:
        ValueError: If the row id list is empty.

    Side effects:
        Copies the selected memmap rows into a contiguous in-memory matrix.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty row selection for {indicator_id!r}.")
    signal_matrix = full_signal_matrices_15m[indicator_id]
    if USE_CONTIGUOUS_TIME_SLICE_15M:
        selected = signal_matrix[row_ids, time_slice_start_15m:time_slice_stop_15m]
    else:
        selected = signal_matrix[row_ids][:, time_mask_15m]
    return np.ascontiguousarray(np.asarray(selected, dtype=np.int8))


row_catalogs_15m = {
    indicator_id: build_row_catalog(
        indicator_id=indicator_id,
        sources=meta["sources"],
        param_name=meta["param_name"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in RUN_INDICATORS_15M.items()
}

row_counts_15m = {
    indicator_id: int(matrix.shape[0])
    for indicator_id, matrix in full_signal_matrices_15m.items()
}
raw_variant_count_2 = math.prod(row_counts_15m.values())
print("two indicator row counts:", row_counts_15m)
print("plain two-indicator combinations:", raw_variant_count_2)
print("sample ma.hma rows:", row_catalogs_15m["ma.hma"][:3])


In [ ]:
def topk_fraction_idx(score: np.ndarray, frac: float) -> np.ndarray:
    """Select top indices by fraction from a one-dimensional score vector.

    Parameters:
        score: One-dimensional score array.
        frac: Fraction in `(0, 1]` describing how many elements to keep.

    Returns:
        Int32 indices of the kept elements.

    Assumptions:
        The input contains at least one finite element.

    Raises:
        ValueError: If `frac` is outside `(0, 1]`.

    Side effects:
        None.
    """
    if not (0.0 < frac <= 1.0):
        raise ValueError(f"frac must be in (0, 1], got {frac!r}")
    n = int(score.shape[0])
    k = max(1, int(math.ceil(n * frac)))
    if k >= n:
        return np.arange(n, dtype=np.int32)
    idx = np.argpartition(score, n - k)[n - k:]
    return np.sort(idx.astype(np.int32))


def single_score_chunked(sig_T_i8: np.ndarray, ret_f32: np.ndarray, chunk: int) -> np.ndarray:
    """Compute a chunked dot-product proxy score for one indicator family.

    Parameters:
        sig_T_i8: Int8 signal matrix shaped `(n_rows, n_intervals)`.
        ret_f32: Float32 return vector shaped `(n_intervals,)`.
        chunk: Chunk length along the time axis.

    Returns:
        Float32 proxy score per row.

    Assumptions:
        The signal matrix is already aligned to the return intervals.

    Raises:
        ValueError: If matrix and return lengths disagree.

    Side effects:
        None.
    """
    if sig_T_i8.shape[1] != ret_f32.shape[0]:
        raise ValueError("Signal matrix and return vector must share the same interval length.")
    n_rows, n_intervals = sig_T_i8.shape
    out = np.zeros(n_rows, dtype=np.float32)
    for t0 in range(0, n_intervals, chunk):
        t1 = min(t0 + chunk, n_intervals)
        out += sig_T_i8[:, t0:t1].astype(np.float32) @ ret_f32[t0:t1]
    return out


@njit_cached(parallel=True, fastmath=False)
def fused_row_prefilter_stats(
    trade_T: np.ndarray,
    ret_15m: np.ndarray,
    out_nonzero: np.ndarray,
    out_proxy: np.ndarray,
    out_change_count: np.ndarray,
) -> None:
    """Compute row activity, proxy score, and change count in one pass."""
    n_rows = trade_T.shape[0]
    n_sig = trade_T.shape[1]
    n_intervals = ret_15m.shape[0]

    for row_idx in nb.prange(n_rows):
        nonzero = np.int32(0)
        proxy = np.float32(0.0)
        change_count = np.int32(0)

        for t in range(n_intervals):
            value = trade_T[row_idx, t]
            if value != 0:
                nonzero += 1
                proxy += np.float32(value) * ret_15m[t]
            next_t = t + 1
            if next_t < n_sig and trade_T[row_idx, next_t] != value:
                change_count += 1

        for t in range(n_intervals + 1, n_sig):
            if trade_T[row_idx, t] != trade_T[row_idx, t - 1]:
                change_count += 1

        out_nonzero[row_idx] = nonzero
        out_proxy[row_idx] = proxy
        out_change_count[row_idx] = change_count


@njit_cached(parallel=True)
def fill_signal_segments_i8(
    trade_T: np.ndarray,
    starts: np.ndarray,
    ends: np.ndarray,
    values: np.ndarray,
    counts: np.ndarray,
) -> None:
    """Fill padded segment arrays for a filtered signal matrix."""
    n_rows = trade_T.shape[0]
    n_sig = trade_T.shape[1]

    for row_idx in nb.prange(n_rows):
        segment_idx = 0
        segment_start = np.int32(0)
        current_value = trade_T[row_idx, 0]
        for t in range(1, n_sig):
            value = trade_T[row_idx, t]
            if value != current_value:
                starts[row_idx, segment_idx] = segment_start
                ends[row_idx, segment_idx] = np.int32(t)
                values[row_idx, segment_idx] = current_value
                segment_idx += 1
                segment_start = np.int32(t)
                current_value = value

        starts[row_idx, segment_idx] = segment_start
        ends[row_idx, segment_idx] = np.int32(n_sig)
        values[row_idx, segment_idx] = current_value
        counts[row_idx] = np.int32(segment_idx + 1)


def build_signal_segments(trade_T: np.ndarray):
    """Build padded change-point segments for an int8 signal matrix."""
    if trade_T.ndim != 2 or trade_T.shape[1] == 0:
        raise ValueError("trade_T must be a non-empty 2D signal matrix.")
    change_count = (trade_T[:, 1:] != trade_T[:, :-1]).sum(axis=1).astype(np.int32)
    counts_expected = change_count + np.int32(1)
    max_segments = int(counts_expected.max())
    starts = np.zeros((trade_T.shape[0], max_segments), dtype=np.int32)
    ends = np.zeros((trade_T.shape[0], max_segments), dtype=np.int32)
    values = np.zeros((trade_T.shape[0], max_segments), dtype=np.int8)
    counts = np.zeros(trade_T.shape[0], dtype=np.int32)
    fill_signal_segments_i8(trade_T, starts, ends, values, counts)
    if not np.array_equal(counts, counts_expected):
        raise AssertionError("Segment count mismatch while compressing signals.")
    return {
        "starts": starts,
        "ends": ends,
        "values": values,
        "counts": counts,
        "change_count": change_count,
    }


def prefilter_indicator_rows(*, trade_T: np.ndarray, indicator_id: str, row_ids: np.ndarray, top_frac: float, min_nonzero: int, fee_rate: float, time_chunk: int):
    """Apply a fused single-indicator prefilter before exact combo evaluation.

    Parameters:
        trade_T: Int8 signal matrix shaped `(n_rows, n_signal_bars)`.
        indicator_id: Indicator id for diagnostics.
        row_ids: Original artifact row ids aligned to `trade_T` rows.
        top_frac: Fraction of rows to retain after scoring.
        min_nonzero: Minimum number of non-zero signals required to keep a row.
        fee_rate: Per-side fee used as a rough penalty term in the proxy score.
        time_chunk: Kept for call-site compatibility; fused stats scan the row once.

    Returns:
        Standard indicator-pool dictionary without heavy derived artifacts attached yet.

    Assumptions:
        The exact path trades on the next bar, so proxy evaluation uses `trade_T[:, :-1]`.

    Raises:
        ValueError: If no candidate survives the non-zero filter.

    Side effects:
        None.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if trade_T.shape[0] != row_ids.shape[0]:
        raise ValueError(f"Row id alignment mismatch for {indicator_id!r}.")

    nonzero = np.empty(trade_T.shape[0], dtype=np.int32)
    proxy = np.empty(trade_T.shape[0], dtype=np.float32)
    change_count = np.empty(trade_T.shape[0], dtype=np.int32)
    fused_row_prefilter_stats(trade_T, signal_returns_15m, nonzero, proxy, change_count)

    adjusted = proxy - (fee_rate * nonzero.astype(np.float32))
    valid = nonzero >= int(min_nonzero)
    if not np.any(valid):
        raise ValueError(f"No rows survive min_nonzero={min_nonzero} for {indicator_id!r}.")

    valid_idx = np.flatnonzero(valid)
    keep_from_valid = topk_fraction_idx(adjusted[valid_idx], top_frac)
    keep_idx = np.sort(valid_idx[keep_from_valid].astype(np.int32))

    filtered_row_ids = np.ascontiguousarray(row_ids[keep_idx])
    filtered_trade_T = np.ascontiguousarray(trade_T[keep_idx])
    filtered_eval_T = np.ascontiguousarray(filtered_trade_T[:, :n_signal_intervals])
    row_score = np.ascontiguousarray(adjusted[keep_idx])
    filtered_nonzero = np.ascontiguousarray(nonzero[keep_idx])
    filtered_change_count = np.ascontiguousarray(change_count[keep_idx])

    return {
        "indicator_id": indicator_id,
        "row_ids": filtered_row_ids,
        "filtered_row_ids": filtered_row_ids,
        "trade_T": filtered_trade_T,
        "eval_T": filtered_eval_T,
        "segments": None,
        "row_score": row_score,
        "score_adj": row_score,
        "nonzero": filtered_nonzero,
        "change_count": filtered_change_count,
        "metadata": None,
    }


INDICATOR_POOL_SCHEMA_KEYS = (
    "indicator_id",
    "row_ids",
    "trade_T",
    "eval_T",
    "segments",
    "row_score",
    "nonzero",
    "change_count",
    "metadata",
)


def attach_indicator_pool_artifacts(pool: dict) -> dict:
    """Attach universal per-row metadata and compressed signal segments to an indicator pool."""
    indicator_id = pool["indicator_id"]
    segments = build_signal_segments(pool["trade_T"])
    if not np.array_equal(segments["change_count"], pool["change_count"]):
        raise AssertionError(f"Segment change counts differ from fused row stats for {indicator_id!r}.")
    pool["segments"] = segments
    pool["metadata"] = [
        row_catalogs_15m[indicator_id][int(row_id)]
        for row_id in pool["row_ids"]
    ]
    validate_indicator_pool_schema(pool)
    return pool


def validate_indicator_pool_schema(pool: dict) -> None:
    """Validate the standard indicator-pool shape used by search and exact backends."""
    missing = [key for key in INDICATOR_POOL_SCHEMA_KEYS if key not in pool]
    if missing:
        raise KeyError(f"Indicator pool {pool.get('indicator_id')!r} is missing keys: {missing}")

    n_rows = int(pool["trade_T"].shape[0])
    if pool["row_ids"].shape[0] != n_rows:
        raise ValueError(f"row_ids length mismatch for {pool['indicator_id']!r}.")
    if pool["eval_T"].shape[0] != n_rows or pool["eval_T"].shape[1] != n_signal_intervals:
        raise ValueError(f"eval_T shape mismatch for {pool['indicator_id']!r}.")
    if pool["row_score"].shape[0] != n_rows:
        raise ValueError(f"row_score length mismatch for {pool['indicator_id']!r}.")
    if pool["nonzero"].shape[0] != n_rows:
        raise ValueError(f"nonzero length mismatch for {pool['indicator_id']!r}.")
    if pool["change_count"].shape[0] != n_rows:
        raise ValueError(f"change_count length mismatch for {pool['indicator_id']!r}.")
    if len(pool["metadata"]) != n_rows:
        raise ValueError(f"metadata length mismatch for {pool['indicator_id']!r}.")

    segments = pool["segments"]
    for key in ("starts", "ends", "values", "counts", "change_count"):
        if key not in segments:
            raise KeyError(f"segments for {pool['indicator_id']!r} are missing {key!r}.")
    if segments["starts"].shape[0] != n_rows or segments["ends"].shape[0] != n_rows or segments["values"].shape[0] != n_rows:
        raise ValueError(f"segment matrix row mismatch for {pool['indicator_id']!r}.")
    if segments["counts"].shape[0] != n_rows:
        raise ValueError(f"segment counts length mismatch for {pool['indicator_id']!r}.")


def prepare_indicator_pool(*, indicator_id: str, row_ids: np.ndarray | None = None, top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, fee_rate: float = FEE_RATE, time_chunk: int = TIME_CHUNK):
    """Load one indicator into the standard pool format used by search backends.

    Parameters:
        indicator_id: Indicator id to prepare.
        row_ids: Optional original artifact row ids. When omitted, all rows are used.
        top_frac: Fraction of rows to keep after proxy scoring.
        min_nonzero: Minimum count of non-zero signals required for a row.
        fee_rate: Fee penalty used inside the proxy prefilter.
        time_chunk: Kept for compatibility with older chunked proxy scoring.

    Returns:
        Standard indicator-pool dictionary with row ids, matrices, segments, stats, and metadata.

    Assumptions:
        Row metadata is available in `row_catalogs_15m`.

    Raises:
        ValueError: If the initial row selection is empty.

    Side effects:
        Loads the selected signal rows into memory and builds compressed signal segments.
    """
    if row_ids is None:
        row_ids = np.arange(row_counts_15m[indicator_id], dtype=np.int32)
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty requested pool for {indicator_id!r}.")
    trade_T = extract_signal_rows(indicator_id=indicator_id, row_ids=row_ids)
    pool = prefilter_indicator_rows(
        trade_T=trade_T,
        indicator_id=indicator_id,
        row_ids=row_ids,
        top_frac=top_frac,
        min_nonzero=min_nonzero,
        fee_rate=fee_rate,
        time_chunk=time_chunk,
    )
    return attach_indicator_pool_artifacts(pool)


def prepare_indicator_pools(*, indicator_ids: tuple[str, ...], row_pools: dict, top_frac: float, min_nonzero: int, fee_rate: float, time_chunk: int) -> dict:
    """Prepare multiple indicators into the same standard pool schema."""
    indicator_pools = {}
    for indicator_id in indicator_ids:
        requested_rows = np.asarray(row_pools[indicator_id], dtype=np.int32)
        indicator_pools[indicator_id] = prepare_indicator_pool(
            indicator_id=indicator_id,
            row_ids=requested_rows,
            top_frac=top_frac,
            min_nonzero=min_nonzero,
            fee_rate=fee_rate,
            time_chunk=time_chunk,
        )
    return indicator_pools


In [ ]:
@njit_cached(inline="always")
def consensus_dir2(dema_value: np.int8, hma_value: np.int8) -> np.int8:
    """Resolve the two-indicator consensus direction for one signal bar.

    Parameters:
        dema_value: DEMA signal on one bar.
        hma_value: HMA signal on one bar.

    Returns:
        `1` for unanimous long, `-1` for unanimous short, `0` otherwise.

    Assumptions:
        Inputs are already normalized to `{-1, 0, 1}`.

    Raises:
        None.

    Side effects:
        None.
    """
    if dema_value == 1 and hma_value == 1:
        return np.int8(1)
    if dema_value == -1 and hma_value == -1:
        return np.int8(-1)
    return np.int8(0)


@njit_cached()
def build_trade_list_for_two_rows(
    dema_sig_row: np.ndarray,
    hma_sig_row: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_entry_exec_idx: np.ndarray,
    out_dir: np.ndarray,
    out_sig_exit_exec_idx: np.ndarray,
) -> np.int32:
    """Build a compact trade list for one two-indicator consensus strategy.

    Parameters:
        dema_sig_row: DEMA signal row over the active 15m run range.
        hma_sig_row: HMA signal row over the active 15m run range.
        sig_entry_exec_idx: Mapping from each 15m signal bar to the next 1m execution entry index.
        T_exec: Exclusive 1m execution limit for the run range.
        out_entry_exec_idx: Preallocated output for trade entry indices.
        out_dir: Preallocated output for trade directions.
        out_sig_exit_exec_idx: Preallocated output for signal-exit indices.

    Returns:
        Number of trades written into the output buffers, or `-1` when capacity is insufficient.

    Assumptions:
        Repeated confirmations in the same direction while already in a position are ignored.

    Raises:
        None directly; buffer overflow is reported via `-1`.

    Side effects:
        Mutates the provided output arrays in place.
    """
    n_sig = dema_sig_row.shape[0]
    n_trades = np.int32(0)
    current_dir = np.int8(0)
    current_entry = np.int32(0)

    for t in range(n_sig):
        dirn = consensus_dir2(dema_sig_row[t], hma_sig_row[t])
        if dirn == 0:
            continue

        entry_exec = sig_entry_exec_idx[t]
        if entry_exec >= T_exec:
            break

        if current_dir == 0:
            current_dir = dirn
            current_entry = np.int32(entry_exec)
            continue

        if dirn == current_dir:
            continue

        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)

        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = np.int32(entry_exec)
        n_trades += 1
        current_dir = dirn
        current_entry = np.int32(entry_exec)

    if current_dir != 0:
        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)
        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = T_exec
        n_trades += 1

    return n_trades


@njit_cached(parallel=True)
def count_trades_for_two_combos(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_trade_counts: np.ndarray,
) -> None:
    """Count compressed trades for a chunk of two-indicator combinations.

    Parameters:
        combo_dema_idx: Row indices into the filtered DEMA trade matrix.
        combo_hma_idx: Row indices into the filtered HMA trade matrix.
        dema_trade_T: Filtered DEMA trade matrix shaped `(n_rows, n_signal_bars)`.
        hma_trade_T: Filtered HMA trade matrix shaped `(n_rows, n_signal_bars)`.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        T_exec: Exclusive 1m execution limit for the run range.
        out_trade_counts: Output vector for trade counts per combination.

    Returns:
        None.

    Assumptions:
        Both combo index arrays share the same length.

    Raises:
        None.

    Side effects:
        Writes counts into `out_trade_counts`.
    """
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        current_dir = np.int8(0)
        n_trades = np.int32(0)
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]

        for t in range(n_sig):
            dirn = consensus_dir2(dema_trade_T[di, t], hma_trade_T[hi, t])
            if dirn == 0:
                continue
            if sig_entry_exec_idx[t] >= T_exec:
                break
            if current_dir == 0:
                current_dir = dirn
                continue
            if dirn != current_dir:
                n_trades += 1
                current_dir = dirn

        if current_dir != 0:
            n_trades += 1
        out_trade_counts[k] = n_trades


In [ ]:
@njit_cached(inline="always")
def trade_sharpe_kernel(trade_count: np.int32, sum_trade_return: float, sum_trade_return_squared: float, bars_per_year_exec: float, sentinel_index: np.int32) -> float:
    """Compute trade-level Sharpe from accumulated returns.

    Parameters:
        trade_count: Number of closed trades.
        sum_trade_return: Sum of per-trade decimal returns.
        sum_trade_return_squared: Sum of squared per-trade decimal returns.
        bars_per_year_exec: Annualization denominator in execution bars.
        sentinel_index: Total execution bars in the replay window.

    Returns:
        Trade-level Sharpe ratio.

    Assumptions:
        The caller already filtered invalid or empty trade sets.

    Raises:
        None.

    Side effects:
        None.
    """
    if trade_count <= 1:
        return 0.0
    mean_trade_return = sum_trade_return / float(trade_count)
    variance = (sum_trade_return_squared / float(trade_count)) - (mean_trade_return * mean_trade_return)
    if variance <= 0.0:
        return 0.0
    years = float(sentinel_index) / float(bars_per_year_exec)
    if years <= 0.0:
        years = 1.0
    trades_per_year = float(trade_count) / years
    return (mean_trade_return / math.sqrt(variance)) * math.sqrt(trades_per_year)


@njit_cached(inline="always")
def score_trade_list_no_risk(
    entry_exec_idx: np.ndarray,
    dir_arr: np.ndarray,
    sig_exit_exec_idx: np.ndarray,
    n_trades: np.int32,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
) -> tuple[float, float, float, float, np.int32, float, float, float, float, float]:
    """Score one compact trade list with no-risk execution semantics.

    Parameters:
        entry_exec_idx: Entry execution indexes for one combo.
        dir_arr: Trade directions for one combo.
        sig_exit_exec_idx: Signal-exit execution indexes for one combo.
        n_trades: Number of valid trades.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.
        init_cash_quote: Initial quote balance.
        fixed_quote: Fixed quote size when fixed sizing is enabled.
        fee_rate: Decimal per-side fee rate.
        slippage_rate: Decimal slippage rate.
        safe_profit_percent: Profit-lock percentage.
        use_fixed_quote: Whether fixed quote sizing is enabled.
        use_profit_lock: Whether profit lock is enabled.
        bars_per_year_exec: Annualization denominator in execution bars.
        close_on_end: Whether a final open trade closes at the last execution close.

    Returns:
        Tuple of no-risk metrics aligned with the persisted summary metrics contract.

    Assumptions:
        Long entries buy above the open by slippage, short entries sell below the open by slippage.

    Raises:
        None.

    Side effects:
        None.
    """
    available_quote = init_cash_quote
    safe_quote = 0.0
    equity = init_cash_quote
    peak_equity = equity
    max_drawdown_pct = 0.0
    gross_profit_quote = 0.0
    gross_loss_quote = 0.0
    closed_trade_count = np.int32(0)
    win_count = np.int32(0)
    sum_trade_return = 0.0
    sum_trade_return_squared = 0.0
    total_trade_return_pct = 0.0
    total_trade_exec_bars = 0.0
    exposure_bars = 0.0

    for trade_index in range(n_trades):
        entry_idx = np.int32(entry_exec_idx[trade_index])
        if entry_idx >= T_exec:
            continue
        exit_idx = np.int32(sig_exit_exec_idx[trade_index])
        if exit_idx < T_exec:
            exit_exec_idx = exit_idx
            exit_price_raw = float(exec_open_1m[exit_exec_idx])
        elif close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            exit_price_raw = float(exec_close_1m[exit_exec_idx])
        else:
            continue

        if available_quote <= 0.0:
            continue
        quote_amount = available_quote
        if use_fixed_quote == 1 and fixed_quote < quote_amount:
            quote_amount = fixed_quote
        if quote_amount <= 0.0:
            continue

        trade_direction = np.int8(dir_arr[trade_index])
        entry_price_raw = float(exec_open_1m[entry_idx])
        if trade_direction == 1:
            entry_fill_price = entry_price_raw * (1.0 + slippage_rate)
            exit_fill_price = exit_price_raw * (1.0 - slippage_rate)
        else:
            entry_fill_price = entry_price_raw * (1.0 - slippage_rate)
            exit_fill_price = exit_price_raw * (1.0 + slippage_rate)

        qty_base = quote_amount / entry_fill_price
        entry_fee_quote = quote_amount * fee_rate
        available_quote -= quote_amount + entry_fee_quote

        exit_quote_amount = qty_base * exit_fill_price
        exit_fee_quote = exit_quote_amount * fee_rate
        if trade_direction == 1:
            gross_pnl_quote = exit_quote_amount - quote_amount
        else:
            gross_pnl_quote = quote_amount - exit_quote_amount
        available_quote += quote_amount + gross_pnl_quote - exit_fee_quote
        net_pnl_quote = gross_pnl_quote - entry_fee_quote - exit_fee_quote

        if use_profit_lock == 1 and net_pnl_quote > 0.0:
            locked_profit_quote = net_pnl_quote * (safe_profit_percent / 100.0)
            available_quote -= locked_profit_quote
            safe_quote += locked_profit_quote

        equity = available_quote + safe_quote
        if equity > peak_equity:
            peak_equity = equity
        elif peak_equity > 0.0:
            drawdown_pct = ((peak_equity - equity) / peak_equity) * 100.0
            if drawdown_pct > max_drawdown_pct:
                max_drawdown_pct = drawdown_pct

        trade_return_pct = (net_pnl_quote / quote_amount) * 100.0
        trade_return = net_pnl_quote / quote_amount
        bars_held = float(exit_exec_idx - entry_idx)
        if bars_held < 0.0:
            bars_held = 0.0

        closed_trade_count += 1
        if net_pnl_quote > 0.0:
            win_count += 1
            gross_profit_quote += net_pnl_quote
        elif net_pnl_quote < 0.0:
            gross_loss_quote += abs(net_pnl_quote)
        sum_trade_return += trade_return
        sum_trade_return_squared += trade_return * trade_return
        total_trade_return_pct += trade_return_pct
        total_trade_exec_bars += bars_held
        exposure_bars += bars_held

    total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
    if gross_loss_quote > 0.0:
        profit_factor = gross_profit_quote / gross_loss_quote
    elif gross_profit_quote > 0.0:
        profit_factor = np.inf
    else:
        profit_factor = 0.0

    if max_drawdown_pct > 0.0:
        return_over_max_drawdown = total_return_pct / max_drawdown_pct
    elif total_return_pct > 0.0:
        return_over_max_drawdown = np.inf
    else:
        return_over_max_drawdown = 0.0

    if closed_trade_count > 0:
        win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
        avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
        avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
    else:
        win_rate_pct = 0.0
        avg_trade_ret_pct = 0.0
        avg_trade_exec_bars = 0.0

    exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
    sharpe_trades = trade_sharpe_kernel(closed_trade_count, sum_trade_return, sum_trade_return_squared, bars_per_year_exec, T_exec)
    return (
        total_return_pct,
        max_drawdown_pct,
        return_over_max_drawdown,
        profit_factor,
        closed_trade_count,
        sharpe_trades,
        win_rate_pct,
        avg_trade_ret_pct,
        avg_trade_exec_bars,
        exposure_pct,
    )


@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_trade_list_fast_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    trade_counts: np.ndarray,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score a chunk of two-indicator combos with no-risk execution metrics.

    Parameters:
        combo_dema_idx: Local DEMA row indices.
        combo_hma_idx: Local HMA row indices.
        dema_trade_T: Filtered DEMA trade matrix.
        hma_trade_T: Filtered HMA trade matrix.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.
        trade_counts: Precomputed compact trade counts per combo.
        init_cash_quote: Initial quote balance.
        fixed_quote: Fixed quote size when fixed sizing is enabled.
        fee_rate: Decimal per-side fee rate.
        slippage_rate: Decimal slippage rate.
        safe_profit_percent: Profit-lock percentage.
        use_fixed_quote: Whether fixed quote sizing is enabled.
        use_profit_lock: Whether profit lock is enabled.
        bars_per_year_exec: Annualization denominator in execution bars.
        close_on_end: Whether a final open trade closes at the last execution close.
        out_*: Output arrays for the scored metrics.

    Returns:
        None.

    Assumptions:
        Each combo is independent and can be scored in parallel.

    Raises:
        None.

    Side effects:
        Writes metrics into the output arrays.
    """
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        alloc_n = np.int32(trade_counts[k])
        if alloc_n <= 0:
            out_total_return_pct[k] = 0.0
            out_max_drawdown_pct[k] = 0.0
            out_return_over_max_drawdown[k] = 0.0
            out_profit_factor[k] = 0.0
            out_trade_count[k] = 0
            out_sharpe_trades[k] = 0.0
            out_win_rate_pct[k] = 0.0
            out_avg_trade_ret_pct[k] = 0.0
            out_avg_trade_exec_bars[k] = 0.0
            out_exposure_pct[k] = 0.0
            continue

        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        entry_arr = np.empty(alloc_n, dtype=np.int32)
        dir_arr = np.empty(alloc_n, dtype=np.int8)
        sig_exit_arr = np.empty(alloc_n, dtype=np.int32)
        n_trades = build_trade_list_for_two_rows(
            dema_trade_T[di],
            hma_trade_T[hi],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )
        if n_trades <= 0:
            out_total_return_pct[k] = 0.0
            out_max_drawdown_pct[k] = 0.0
            out_return_over_max_drawdown[k] = 0.0
            out_profit_factor[k] = 0.0
            out_trade_count[k] = 0
            out_sharpe_trades[k] = 0.0
            out_win_rate_pct[k] = 0.0
            out_avg_trade_ret_pct[k] = 0.0
            out_avg_trade_exec_bars[k] = 0.0
            out_exposure_pct[k] = 0.0
            continue

        metrics = score_trade_list_no_risk(
            entry_arr,
            dir_arr,
            sig_exit_arr,
            n_trades,
            exec_open_1m,
            exec_close_1m,
            T_exec,
            init_cash_quote,
            fixed_quote,
            fee_rate,
            slippage_rate,
            safe_profit_percent,
            use_fixed_quote,
            use_profit_lock,
            bars_per_year_exec,
            close_on_end,
        )
        out_total_return_pct[k] = metrics[0]
        out_max_drawdown_pct[k] = metrics[1]
        out_return_over_max_drawdown[k] = metrics[2]
        out_profit_factor[k] = metrics[3]
        out_trade_count[k] = metrics[4]
        out_sharpe_trades[k] = metrics[5]
        out_win_rate_pct[k] = metrics[6]
        out_avg_trade_ret_pct[k] = metrics[7]
        out_avg_trade_exec_bars[k] = metrics[8]
        out_exposure_pct[k] = metrics[9]



@njit_cached(inline="always")
def apply_no_risk_trade_to_state(
    entry_idx: np.int32,
    trade_direction: np.int8,
    exit_exec_idx: np.int32,
    exit_price_raw: float,
    exec_open_1m: np.ndarray,
    available_quote: float,
    safe_quote: float,
    equity: float,
    peak_equity: float,
    max_drawdown_pct: float,
    gross_profit_quote: float,
    gross_loss_quote: float,
    closed_trade_count: np.int32,
    win_count: np.int32,
    sum_trade_return: float,
    sum_trade_return_squared: float,
    total_trade_return_pct: float,
    total_trade_exec_bars: float,
    exposure_bars: float,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
) -> tuple[float, float, float, float, float, float, float, np.int32, np.int32, float, float, float, float, float]:
    """Apply one closed no-risk trade directly to aggregate metric state."""
    if available_quote <= 0.0:
        return (
            available_quote,
            safe_quote,
            equity,
            peak_equity,
            max_drawdown_pct,
            gross_profit_quote,
            gross_loss_quote,
            closed_trade_count,
            win_count,
            sum_trade_return,
            sum_trade_return_squared,
            total_trade_return_pct,
            total_trade_exec_bars,
            exposure_bars,
        )

    quote_amount = available_quote
    if use_fixed_quote == 1 and fixed_quote < quote_amount:
        quote_amount = fixed_quote
    if quote_amount <= 0.0:
        return (
            available_quote,
            safe_quote,
            equity,
            peak_equity,
            max_drawdown_pct,
            gross_profit_quote,
            gross_loss_quote,
            closed_trade_count,
            win_count,
            sum_trade_return,
            sum_trade_return_squared,
            total_trade_return_pct,
            total_trade_exec_bars,
            exposure_bars,
        )

    entry_price_raw = float(exec_open_1m[entry_idx])
    if trade_direction == 1:
        entry_fill_price = entry_price_raw * (1.0 + slippage_rate)
        exit_fill_price = exit_price_raw * (1.0 - slippage_rate)
    else:
        entry_fill_price = entry_price_raw * (1.0 - slippage_rate)
        exit_fill_price = exit_price_raw * (1.0 + slippage_rate)

    qty_base = quote_amount / entry_fill_price
    entry_fee_quote = quote_amount * fee_rate
    available_quote -= quote_amount + entry_fee_quote

    exit_quote_amount = qty_base * exit_fill_price
    exit_fee_quote = exit_quote_amount * fee_rate
    if trade_direction == 1:
        gross_pnl_quote = exit_quote_amount - quote_amount
    else:
        gross_pnl_quote = quote_amount - exit_quote_amount
    available_quote += quote_amount + gross_pnl_quote - exit_fee_quote
    net_pnl_quote = gross_pnl_quote - entry_fee_quote - exit_fee_quote

    if use_profit_lock == 1 and net_pnl_quote > 0.0:
        locked_profit_quote = net_pnl_quote * (safe_profit_percent / 100.0)
        available_quote -= locked_profit_quote
        safe_quote += locked_profit_quote

    equity = available_quote + safe_quote
    if equity > peak_equity:
        peak_equity = equity
    elif peak_equity > 0.0:
        drawdown_pct = ((peak_equity - equity) / peak_equity) * 100.0
        if drawdown_pct > max_drawdown_pct:
            max_drawdown_pct = drawdown_pct

    trade_return_pct = (net_pnl_quote / quote_amount) * 100.0
    trade_return = net_pnl_quote / quote_amount
    bars_held = float(exit_exec_idx - entry_idx)
    if bars_held < 0.0:
        bars_held = 0.0

    closed_trade_count += 1
    if net_pnl_quote > 0.0:
        win_count += 1
        gross_profit_quote += net_pnl_quote
    elif net_pnl_quote < 0.0:
        gross_loss_quote += abs(net_pnl_quote)
    sum_trade_return += trade_return
    sum_trade_return_squared += trade_return * trade_return
    total_trade_return_pct += trade_return_pct
    total_trade_exec_bars += bars_held
    exposure_bars += bars_held

    return (
        available_quote,
        safe_quote,
        equity,
        peak_equity,
        max_drawdown_pct,
        gross_profit_quote,
        gross_loss_quote,
        closed_trade_count,
        win_count,
        sum_trade_return,
        sum_trade_return_squared,
        total_trade_return_pct,
        total_trade_exec_bars,
        exposure_bars,
    )


@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_streaming_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score two-indicator combos in one pass without a separate trade-count pass or trade-list arrays."""
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        available_quote = init_cash_quote
        safe_quote = 0.0
        equity = init_cash_quote
        peak_equity = equity
        max_drawdown_pct = 0.0
        gross_profit_quote = 0.0
        gross_loss_quote = 0.0
        closed_trade_count = np.int32(0)
        win_count = np.int32(0)
        sum_trade_return = 0.0
        sum_trade_return_squared = 0.0
        total_trade_return_pct = 0.0
        total_trade_exec_bars = 0.0
        exposure_bars = 0.0
        current_dir = np.int8(0)
        current_entry = np.int32(0)

        for t in range(n_sig):
            dirn = consensus_dir2(dema_trade_T[di, t], hma_trade_T[hi, t])
            if dirn == 0:
                continue
            entry_exec = sig_entry_exec_idx[t]
            if entry_exec >= T_exec:
                break
            if current_dir == 0:
                current_dir = dirn
                current_entry = np.int32(entry_exec)
                continue
            if dirn == current_dir:
                continue

            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                np.int32(entry_exec),
                float(exec_open_1m[entry_exec]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )
            current_dir = dirn
            current_entry = np.int32(entry_exec)

        if current_dir != 0 and close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                exit_exec_idx,
                float(exec_close_1m[exit_exec_idx]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )

        total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
        if gross_loss_quote > 0.0:
            profit_factor = gross_profit_quote / gross_loss_quote
        elif gross_profit_quote > 0.0:
            profit_factor = np.inf
        else:
            profit_factor = 0.0

        if max_drawdown_pct > 0.0:
            return_over_max_drawdown = total_return_pct / max_drawdown_pct
        elif total_return_pct > 0.0:
            return_over_max_drawdown = np.inf
        else:
            return_over_max_drawdown = 0.0

        if closed_trade_count > 0:
            win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
            avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
            avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
        else:
            win_rate_pct = 0.0
            avg_trade_ret_pct = 0.0
            avg_trade_exec_bars = 0.0

        exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
        sharpe_trades = trade_sharpe_kernel(
            closed_trade_count,
            sum_trade_return,
            sum_trade_return_squared,
            bars_per_year_exec,
            T_exec,
        )
        out_total_return_pct[k] = total_return_pct
        out_max_drawdown_pct[k] = max_drawdown_pct
        out_return_over_max_drawdown[k] = return_over_max_drawdown
        out_profit_factor[k] = profit_factor
        out_trade_count[k] = closed_trade_count
        out_sharpe_trades[k] = sharpe_trades
        out_win_rate_pct[k] = win_rate_pct
        out_avg_trade_ret_pct[k] = avg_trade_ret_pct
        out_avg_trade_exec_bars[k] = avg_trade_exec_bars
        out_exposure_pct[k] = exposure_pct




@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_event_segments_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_segment_starts: np.ndarray,
    dema_segment_ends: np.ndarray,
    dema_segment_values: np.ndarray,
    dema_segment_counts: np.ndarray,
    hma_segment_starts: np.ndarray,
    hma_segment_ends: np.ndarray,
    hma_segment_values: np.ndarray,
    hma_segment_counts: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score two-indicator combos by merging compressed signal segments instead of scanning every bar."""
    K = combo_dema_idx.shape[0]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        available_quote = init_cash_quote
        safe_quote = 0.0
        equity = init_cash_quote
        peak_equity = equity
        max_drawdown_pct = 0.0
        gross_profit_quote = 0.0
        gross_loss_quote = 0.0
        closed_trade_count = np.int32(0)
        win_count = np.int32(0)
        sum_trade_return = 0.0
        sum_trade_return_squared = 0.0
        total_trade_return_pct = 0.0
        total_trade_exec_bars = 0.0
        exposure_bars = 0.0
        current_dir = np.int8(0)
        current_entry = np.int32(0)
        dema_segment_idx = 0
        hma_segment_idx = 0

        while dema_segment_idx < dema_segment_counts[di] and hma_segment_idx < hma_segment_counts[hi]:
            dema_start = dema_segment_starts[di, dema_segment_idx]
            dema_end = dema_segment_ends[di, dema_segment_idx]
            hma_start = hma_segment_starts[hi, hma_segment_idx]
            hma_end = hma_segment_ends[hi, hma_segment_idx]
            segment_start = dema_start if dema_start >= hma_start else hma_start
            segment_end = dema_end if dema_end <= hma_end else hma_end

            if segment_start < segment_end:
                dirn = consensus_dir2(
                    dema_segment_values[di, dema_segment_idx],
                    hma_segment_values[hi, hma_segment_idx],
                )
                if dirn != 0:
                    entry_exec = sig_entry_exec_idx[segment_start]
                    if entry_exec >= T_exec:
                        break
                    if current_dir == 0:
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)
                    elif dirn != current_dir:
                        (
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                        ) = apply_no_risk_trade_to_state(
                            current_entry,
                            current_dir,
                            np.int32(entry_exec),
                            float(exec_open_1m[entry_exec]),
                            exec_open_1m,
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                            init_cash_quote,
                            fixed_quote,
                            fee_rate,
                            slippage_rate,
                            safe_profit_percent,
                            use_fixed_quote,
                            use_profit_lock,
                        )
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)

            if dema_end == segment_end:
                dema_segment_idx += 1
            if hma_end == segment_end:
                hma_segment_idx += 1

        if current_dir != 0 and close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                exit_exec_idx,
                float(exec_close_1m[exit_exec_idx]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )

        total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
        if gross_loss_quote > 0.0:
            profit_factor = gross_profit_quote / gross_loss_quote
        elif gross_profit_quote > 0.0:
            profit_factor = np.inf
        else:
            profit_factor = 0.0

        if max_drawdown_pct > 0.0:
            return_over_max_drawdown = total_return_pct / max_drawdown_pct
        elif total_return_pct > 0.0:
            return_over_max_drawdown = np.inf
        else:
            return_over_max_drawdown = 0.0

        if closed_trade_count > 0:
            win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
            avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
            avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
        else:
            win_rate_pct = 0.0
            avg_trade_ret_pct = 0.0
            avg_trade_exec_bars = 0.0

        exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
        sharpe_trades = trade_sharpe_kernel(
            closed_trade_count,
            sum_trade_return,
            sum_trade_return_squared,
            bars_per_year_exec,
            T_exec,
        )
        out_total_return_pct[k] = total_return_pct
        out_max_drawdown_pct[k] = max_drawdown_pct
        out_return_over_max_drawdown[k] = return_over_max_drawdown
        out_profit_factor[k] = profit_factor
        out_trade_count[k] = closed_trade_count
        out_sharpe_trades[k] = sharpe_trades
        out_win_rate_pct[k] = win_rate_pct
        out_avg_trade_ret_pct[k] = avg_trade_ret_pct
        out_avg_trade_exec_bars[k] = avg_trade_exec_bars
        out_exposure_pct[k] = exposure_pct


def evaluate_no_risk_trade_list_slow_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
):
    """Reference no-risk evaluation for a small subset of two-indicator combos.

    Parameters:
        combo_dema_idx: Local DEMA row indices.
        combo_hma_idx: Local HMA row indices.
        dema_trade_T: Filtered DEMA trade matrix.
        hma_trade_T: Filtered HMA trade matrix.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.

    Returns:
        List of metric tuples aligned to the input combo order.

    Assumptions:
        Used only by the notebook self-check on a tiny subset.

    Raises:
        None.

    Side effects:
        None.
    """
    results = []
    n_sig = dema_trade_T.shape[1]
    for di, hi in zip(combo_dema_idx.tolist(), combo_hma_idx.tolist()):
        entry_arr = np.empty(n_sig, dtype=np.int32)
        dir_arr = np.empty(n_sig, dtype=np.int8)
        sig_exit_arr = np.empty(n_sig, dtype=np.int32)
        n_trades = int(build_trade_list_for_two_rows(
            dema_trade_T[int(di)],
            hma_trade_T[int(hi)],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        ))
        if n_trades <= 0:
            results.append((0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0.0, 0.0, 0.0))
            continue
        results.append(
            score_trade_list_no_risk(
                entry_arr,
                dir_arr,
                sig_exit_arr,
                np.int32(n_trades),
                exec_open_1m,
                exec_close_1m,
                T_exec,
                INIT_CASH_QUOTE,
                FIXED_QUOTE,
                FEE_RATE,
                SLIPPAGE_RATE,
                SAFE_PROFIT_PERCENT,
                np.int8(1 if USE_FIXED_QUOTE else 0),
                np.int8(1 if USE_PROFIT_LOCK else 0),
                BARS_PER_YEAR_EXEC_1M,
                CLOSE_ON_END,
            )
        )
    return results


In [ ]:
EXACT_BACKEND_AUTO = "auto"
EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK = "event_segments_2_no_risk"
EXACT_BACKEND_STREAMING_2_NO_RISK = "streaming_2_no_risk"


def resolve_no_risk_exact_backend(*, indicator_ids: tuple[str, ...], requested_backend: str = EXACT_BACKEND_AUTO) -> dict:
    """Resolve a no-risk exact backend strategy for the requested indicator arity."""
    indicator_ids = tuple(indicator_ids)
    arity = len(indicator_ids)
    if arity == 2 and requested_backend in (EXACT_BACKEND_AUTO, "event_segments", EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK):
        return {
            "name": EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK,
            "mode": "no_risk",
            "arity": arity,
            "indicator_ids": indicator_ids,
            "requires_segments": True,
        }
    if arity == 2 and requested_backend in ("streaming", EXACT_BACKEND_STREAMING_2_NO_RISK):
        return {
            "name": EXACT_BACKEND_STREAMING_2_NO_RISK,
            "mode": "no_risk",
            "arity": arity,
            "indicator_ids": indicator_ids,
            "requires_segments": False,
        }
    raise NotImplementedError(
        f"No no-risk exact backend for arity={arity}, requested_backend={requested_backend!r}. "
        "Add a specialized backend or a generic fallback before widening this search."
    )


def evaluate_no_risk_exact_chunk_two(
    *,
    selected: dict,
    indicator_pools: dict,
    exact_strategy: dict,
    total_return_pct: np.ndarray,
    max_drawdown_pct: np.ndarray,
    return_over_max_drawdown: np.ndarray,
    profit_factor: np.ndarray,
    trade_count: np.ndarray,
    sharpe_trades: np.ndarray,
    win_rate_pct: np.ndarray,
    avg_trade_ret_pct: np.ndarray,
    avg_trade_exec_bars: np.ndarray,
    exposure_pct: np.ndarray,
) -> None:
    """Dispatch one two-indicator no-risk exact chunk through the selected backend."""
    if exact_strategy["arity"] != 2:
        raise NotImplementedError(f"Two-indicator dispatcher got arity={exact_strategy['arity']!r}.")
    left_id, right_id = exact_strategy["indicator_ids"]

    if exact_strategy["name"] == EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK:
        left_segments = indicator_pools[left_id]["segments"]
        right_segments = indicator_pools[right_id]["segments"]
        evaluate_no_risk_event_segments_two(
            selected[left_id],
            selected[right_id],
            left_segments["starts"],
            left_segments["ends"],
            left_segments["values"],
            left_segments["counts"],
            right_segments["starts"],
            right_segments["ends"],
            right_segments["values"],
            right_segments["counts"],
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            price_fields_1m["close"],
            T_exec_limit_1m,
            INIT_CASH_QUOTE,
            FIXED_QUOTE,
            FEE_RATE,
            SLIPPAGE_RATE,
            SAFE_PROFIT_PERCENT,
            np.int8(1 if USE_FIXED_QUOTE else 0),
            np.int8(1 if USE_PROFIT_LOCK else 0),
            BARS_PER_YEAR_EXEC_1M,
            CLOSE_ON_END,
            total_return_pct,
            max_drawdown_pct,
            return_over_max_drawdown,
            profit_factor,
            trade_count,
            sharpe_trades,
            win_rate_pct,
            avg_trade_ret_pct,
            avg_trade_exec_bars,
            exposure_pct,
        )
        return

    if exact_strategy["name"] == EXACT_BACKEND_STREAMING_2_NO_RISK:
        evaluate_no_risk_streaming_two(
            selected[left_id],
            selected[right_id],
            indicator_pools[left_id]["trade_T"],
            indicator_pools[right_id]["trade_T"],
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            price_fields_1m["close"],
            T_exec_limit_1m,
            INIT_CASH_QUOTE,
            FIXED_QUOTE,
            FEE_RATE,
            SLIPPAGE_RATE,
            SAFE_PROFIT_PERCENT,
            np.int8(1 if USE_FIXED_QUOTE else 0),
            np.int8(1 if USE_PROFIT_LOCK else 0),
            BARS_PER_YEAR_EXEC_1M,
            CLOSE_ON_END,
            total_return_pct,
            max_drawdown_pct,
            return_over_max_drawdown,
            profit_factor,
            trade_count,
            sharpe_trades,
            win_rate_pct,
            avg_trade_ret_pct,
            avg_trade_exec_bars,
            exposure_pct,
        )
        return

    raise NotImplementedError(f"Unsupported exact backend: {exact_strategy['name']!r}.")


@njit_cached(parallel=True, fastmath=True)
def proxy_prefilter_combos_chunk_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_eval_T: np.ndarray,
    hma_eval_T: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
    out_confirm: np.ndarray,
    out_proxy: np.ndarray,
) -> None:
    """Compute cheap confirmation counts and proxy scores for one two-indicator combo chunk.

    Parameters:
        combo_dema_idx: Local DEMA row indices.
        combo_hma_idx: Local HMA row indices.
        dema_eval_T: Filtered DEMA evaluation matrix shaped `(n_rows, n_signal_intervals)`.
        hma_eval_T: Filtered HMA evaluation matrix shaped `(n_rows, n_signal_intervals)`.
        ret_15m: Close-to-close 15m return vector.
        min_confirm: Minimum number of consensus confirmations required to keep a combo.
        fee_penalty_per_confirm: Linear fee penalty applied to the proxy score.
        out_confirm: Output confirmation-count vector.
        out_proxy: Output proxy-score vector.

    Returns:
        None.

    Assumptions:
        Both combo index arrays share the same length.

    Raises:
        None.

    Side effects:
        Writes confirmation counts and proxy scores into the provided arrays.
    """
    K = combo_dema_idx.shape[0]
    n_int = ret_15m.shape[0]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        confirms = np.int32(0)
        proxy = np.float32(0.0)

        for t in range(n_int):
            dirn = consensus_dir2(dema_eval_T[di, t], hma_eval_T[hi, t])
            if dirn == 1:
                confirms += 1
                proxy += ret_15m[t]
            elif dirn == -1:
                confirms += 1
                proxy -= ret_15m[t]

        out_confirm[k] = confirms
        if confirms >= min_confirm:
            out_proxy[k] = proxy - fee_penalty_per_confirm * np.float32(confirms)
        else:
            out_proxy[k] = NEG_INF



@njit_cached(inline="always")
def proxy_for_two_rows(
    dema_eval_row: np.ndarray,
    hma_eval_row: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
) -> tuple[np.int32, np.float32]:
    """Compute confirm count and proxy score for one final two-indicator result."""
    confirms = np.int32(0)
    proxy = np.float32(0.0)
    for t in range(ret_15m.shape[0]):
        dirn = consensus_dir2(dema_eval_row[t], hma_eval_row[t])
        if dirn == 1:
            confirms += 1
            proxy += ret_15m[t]
        elif dirn == -1:
            confirms += 1
            proxy -= ret_15m[t]
    if confirms >= min_confirm:
        return confirms, proxy - fee_penalty_per_confirm * np.float32(confirms)
    return confirms, NEG_INF


def build_combo_proxy_cache_two(*, dema_eval_T: np.ndarray, hma_eval_T: np.ndarray, ret_15m: np.ndarray, min_confirm: int, fee_penalty_per_confirm: np.float32):
    """Build matrix-backed confirm/proxy lookup tables for active combo pruning."""
    ret = ret_15m.astype(np.float32, copy=False)
    dema_pos = (dema_eval_T == 1).astype(np.float32)
    dema_neg = (dema_eval_T == -1).astype(np.float32)
    hma_pos = (hma_eval_T == 1).astype(np.float32)
    hma_neg = (hma_eval_T == -1).astype(np.float32)

    proxy_matrix = dema_pos @ np.ascontiguousarray((hma_pos * ret).T)
    proxy_matrix -= dema_neg @ np.ascontiguousarray((hma_neg * ret).T)
    confirm_matrix = dema_pos @ np.ascontiguousarray(hma_pos.T)
    confirm_matrix += dema_neg @ np.ascontiguousarray(hma_neg.T)
    confirm_matrix = np.rint(confirm_matrix).astype(np.int32)
    proxy_matrix = proxy_matrix.astype(np.float32, copy=False)
    proxy_matrix -= fee_penalty_per_confirm * confirm_matrix.astype(np.float32)
    proxy_matrix[confirm_matrix < int(min_confirm)] = NEG_INF
    return confirm_matrix, proxy_matrix


def gather_combo_proxy_cache_two(*, combo_chunk, combo_proxy_cache):
    """Gather confirm/proxy vectors for one chunk from matrix-backed lookup tables."""
    confirm_matrix, proxy_matrix = combo_proxy_cache
    dema_idx = combo_chunk["ma.dema"]
    hma_idx = combo_chunk["ma.hma"]
    return (
        np.ascontiguousarray(confirm_matrix[dema_idx, hma_idx]),
        np.ascontiguousarray(proxy_matrix[dema_idx, hma_idx]),
    )

def iter_combo_chunks_two(*, local_row_pools, chunk_size: int):
    """Yield bounded combo chunks over filtered local two-indicator row pools.

    Parameters:
        local_row_pools: Mapping of indicator id to local row indices inside filtered matrices.
        chunk_size: Maximum number of combinations per yielded chunk.

    Returns:
        Generator of dictionaries containing aligned combo index arrays.

    Assumptions:
        Both required indicator ids are present in `local_row_pools`.

    Raises:
        None.

    Side effects:
        None.
    """
    required = ("ma.dema", "ma.hma")
    buffers = {indicator_id: [] for indicator_id in required}

    for combo in itertools.product(*(local_row_pools[indicator_id] for indicator_id in required)):
        for indicator_id, value in zip(required, combo):
            buffers[indicator_id].append(int(value))
        if len(buffers[required[0]]) >= chunk_size:
            yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in required}
            buffers = {indicator_id: [] for indicator_id in required}

    if buffers[required[0]]:
        yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in required}


def run_fast_vs_reference_self_check_two(*, combo_chunk, indicator_pools, exact_strategy: dict, check_n: int = SELF_CHECK_N_DEFAULT, ret_tol: float = 1e-4):
    """Compare the selected exact backend against the slow trade-list reference on a small subset.

    Parameters:
        combo_chunk: Dictionary with combo index arrays over filtered local row pools.
        indicator_pools: Prefiltered standard indicator pool dictionaries.
        exact_strategy: Resolved exact backend strategy.
        check_n: Maximum number of combinations to compare.
        ret_tol: Allowed absolute return difference between reference and backend outputs.

    Returns:
        Dictionary with the number of checked combos and maximum absolute return differences.

    Assumptions:
        The chunk already passed combo prefiltering and contains at least one combination.

    Raises:
        AssertionError: If the selected backend differs from the slow reference beyond tolerance.

    Side effects:
        Executes exact reference and selected backend on the requested subset.
    """
    left_id, right_id = exact_strategy["indicator_ids"]
    n_check = min(check_n, int(combo_chunk[left_id].shape[0]))
    if n_check <= 0:
        return {"checked": 0, "max_abs_best_ret_diff": 0.0, "max_abs_exact_backend_ret_diff": 0.0}

    subset = {indicator_id: combo_chunk[indicator_id][:n_check] for indicator_id in exact_strategy["indicator_ids"]}
    trade_counts = np.empty(n_check, dtype=np.int32)
    count_trades_for_two_combos(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        T_exec_limit_1m,
        trade_counts,
    )

    fast_total_return_pct = np.empty(n_check, dtype=np.float64)
    fast_max_drawdown_pct = np.empty(n_check, dtype=np.float64)
    fast_return_over_max_drawdown = np.empty(n_check, dtype=np.float64)
    fast_profit_factor = np.empty(n_check, dtype=np.float64)
    fast_trade_count = np.empty(n_check, dtype=np.int32)
    fast_sharpe_trades = np.empty(n_check, dtype=np.float64)
    fast_win_rate_pct = np.empty(n_check, dtype=np.float64)
    fast_avg_trade_ret_pct = np.empty(n_check, dtype=np.float64)
    fast_avg_trade_exec_bars = np.empty(n_check, dtype=np.float64)
    fast_exposure_pct = np.empty(n_check, dtype=np.float64)

    evaluate_no_risk_trade_list_fast_two(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        price_fields_1m["close"],
        T_exec_limit_1m,
        trade_counts,
        INIT_CASH_QUOTE,
        FIXED_QUOTE,
        FEE_RATE,
        SLIPPAGE_RATE,
        SAFE_PROFIT_PERCENT,
        np.int8(1 if USE_FIXED_QUOTE else 0),
        np.int8(1 if USE_PROFIT_LOCK else 0),
        BARS_PER_YEAR_EXEC_1M,
        CLOSE_ON_END,
        fast_total_return_pct,
        fast_max_drawdown_pct,
        fast_return_over_max_drawdown,
        fast_profit_factor,
        fast_trade_count,
        fast_sharpe_trades,
        fast_win_rate_pct,
        fast_avg_trade_ret_pct,
        fast_avg_trade_exec_bars,
        fast_exposure_pct,
    )

    slow = evaluate_no_risk_trade_list_slow_two(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        price_fields_1m["close"],
        T_exec_limit_1m,
    )
    slow_total_return_pct = np.asarray([row[0] for row in slow], dtype=np.float64)
    slow_trade_count = np.asarray([row[4] for row in slow], dtype=np.int32)

    if not np.array_equal(slow_trade_count, fast_trade_count):
        raise AssertionError("Fast trade-list counts differ from the slow reference.")

    max_abs_best_ret_diff = float(np.max(np.abs(slow_total_return_pct - fast_total_return_pct)))
    if max_abs_best_ret_diff > ret_tol:
        raise AssertionError(
            f"Fast trade-list total return differs from the slow reference by {max_abs_best_ret_diff}, tolerance {ret_tol}."
        )

    backend_total_return_pct = np.empty(n_check, dtype=np.float64)
    backend_max_drawdown_pct = np.empty(n_check, dtype=np.float64)
    backend_return_over_max_drawdown = np.empty(n_check, dtype=np.float64)
    backend_profit_factor = np.empty(n_check, dtype=np.float64)
    backend_trade_count = np.empty(n_check, dtype=np.int32)
    backend_sharpe_trades = np.empty(n_check, dtype=np.float64)
    backend_win_rate_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_ret_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_exec_bars = np.empty(n_check, dtype=np.float64)
    backend_exposure_pct = np.empty(n_check, dtype=np.float64)
    evaluate_no_risk_exact_chunk_two(
        selected=subset,
        indicator_pools=indicator_pools,
        exact_strategy=exact_strategy,
        total_return_pct=backend_total_return_pct,
        max_drawdown_pct=backend_max_drawdown_pct,
        return_over_max_drawdown=backend_return_over_max_drawdown,
        profit_factor=backend_profit_factor,
        trade_count=backend_trade_count,
        sharpe_trades=backend_sharpe_trades,
        win_rate_pct=backend_win_rate_pct,
        avg_trade_ret_pct=backend_avg_trade_ret_pct,
        avg_trade_exec_bars=backend_avg_trade_exec_bars,
        exposure_pct=backend_exposure_pct,
    )
    if not np.array_equal(slow_trade_count, backend_trade_count):
        raise AssertionError(f"Exact backend {exact_strategy['name']!r} trade counts differ from the slow reference.")
    max_abs_exact_backend_ret_diff = float(np.max(np.abs(slow_total_return_pct - backend_total_return_pct)))
    if max_abs_exact_backend_ret_diff > ret_tol:
        raise AssertionError(
            f"Exact backend {exact_strategy['name']!r} total return differs from the slow reference by "
            f"{max_abs_exact_backend_ret_diff}, tolerance {ret_tol}."
        )

    return {
        "checked": n_check,
        "exact_backend": exact_strategy["name"],
        "max_abs_best_ret_diff": max_abs_best_ret_diff,
        "max_abs_exact_backend_ret_diff": max_abs_exact_backend_ret_diff,
    }


def search_topk_two_indicator_no_risk(*, row_pools, indicator_top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, combo_top_frac: float = COMBO_PREFILTER_TOP_FRAC, combo_min_confirm: int = COMBO_MIN_CONFIRM, combo_chunk_size: int = COMBO_CHUNK_SIZE, top_k: int = TOP_K_DEFAULT, self_check_n: int = SELF_CHECK_N_DEFAULT, exact_backend: str = EXACT_BACKEND_AUTO, verbose: bool = True):
    """Run a two-indicator no-risk search using standard pools and a resolved exact backend.

    Parameters:
        row_pools: Mapping of indicator id to original artifact row ids.
        indicator_top_frac: Fraction of rows to keep in the single-indicator prefilter.
        min_nonzero: Minimum number of non-zero signals required for an indicator row.
        combo_top_frac: Fraction of valid combinations to keep inside each combo chunk before exact evaluation.
        combo_min_confirm: Minimum number of consensus confirmations required before exact evaluation.
        combo_chunk_size: Maximum number of combos per chunk.
        top_k: Number of best exact results to retain globally.
        self_check_n: Number of exact combos to validate against the slow reference.
        exact_backend: Exact backend strategy name. `auto` resolves to `event_segments_2_no_risk` for this arity.
        verbose: Whether to print diagnostics during the search.

    Returns:
        Dictionary containing filtered pool diagnostics, exact backend diagnostics, optional self-check diagnostics, and top-k results.

    Assumptions:
        `row_pools` contains bounded row selections. The full cartesian product may still be large, so prefiltering remains mandatory.

    Raises:
        KeyError: If any required indicator pool is missing.
        AssertionError: If the backend self-check fails.

    Side effects:
        Triggers Numba compilation on first use and runs exact kernels over selected combo chunks.
    """
    required = ("ma.dema", "ma.hma")
    exact_strategy = resolve_no_risk_exact_backend(indicator_ids=required, requested_backend=exact_backend)
    indicator_pools = prepare_indicator_pools(
        indicator_ids=required,
        row_pools=row_pools,
        top_frac=indicator_top_frac,
        min_nonzero=min_nonzero,
        fee_rate=FEE_RATE,
        time_chunk=TIME_CHUNK,
    )

    local_row_pools = {
        indicator_id: np.arange(indicator_pools[indicator_id]["trade_T"].shape[0], dtype=np.int32)
        for indicator_id in required
    }

    heap = []
    self_check = None
    total_combo_chunks = 0
    total_exact_candidates = 0
    fee_penalty_per_confirm = np.float32(1.5 * FEE_RATE)
    combo_prefilter_active = combo_top_frac < 1.0 or combo_min_confirm > 1
    combo_proxy_cache = None
    if combo_prefilter_active:
        combo_proxy_cache = build_combo_proxy_cache_two(
            dema_eval_T=indicator_pools["ma.dema"]["eval_T"],
            hma_eval_T=indicator_pools["ma.hma"]["eval_T"],
            ret_15m=signal_returns_15m,
            min_confirm=combo_min_confirm,
            fee_penalty_per_confirm=fee_penalty_per_confirm,
        )

    for combo_chunk in iter_combo_chunks_two(local_row_pools=local_row_pools, chunk_size=combo_chunk_size):
        total_combo_chunks += 1
        chunk_len = int(combo_chunk[required[0]].shape[0])
        if not combo_prefilter_active:
            keep_idx = np.arange(chunk_len, dtype=np.int32)
            selected = {indicator_id: combo_chunk[indicator_id] for indicator_id in required}
            selected_confirm = None
            selected_proxy = None
        else:
            out_confirm, out_proxy = gather_combo_proxy_cache_two(
                combo_chunk=combo_chunk,
                combo_proxy_cache=combo_proxy_cache,
            )
            valid_idx = np.flatnonzero(out_proxy > NEG_INF / 2)
            if valid_idx.size == 0:
                continue
            keep_local = topk_fraction_idx(out_proxy[valid_idx], combo_top_frac)
            keep_idx = np.sort(valid_idx[keep_local].astype(np.int32))
            selected = {indicator_id: combo_chunk[indicator_id][keep_idx] for indicator_id in required}
            selected_confirm = out_confirm[keep_idx]
            selected_proxy = out_proxy[keep_idx]
        total_exact_candidates += int(keep_idx.size)

        if self_check is None and self_check_n > 0:
            self_check = run_fast_vs_reference_self_check_two(
                combo_chunk=selected,
                indicator_pools=indicator_pools,
                exact_strategy=exact_strategy,
                check_n=self_check_n,
            )

        total_return_pct = np.empty(int(keep_idx.size), dtype=np.float64)
        max_drawdown_pct = np.empty(int(keep_idx.size), dtype=np.float64)
        return_over_max_drawdown = np.empty(int(keep_idx.size), dtype=np.float64)
        profit_factor = np.empty(int(keep_idx.size), dtype=np.float64)
        trade_count = np.empty(int(keep_idx.size), dtype=np.int32)
        sharpe_trades = np.empty(int(keep_idx.size), dtype=np.float64)
        win_rate_pct = np.empty(int(keep_idx.size), dtype=np.float64)
        avg_trade_ret_pct = np.empty(int(keep_idx.size), dtype=np.float64)
        avg_trade_exec_bars = np.empty(int(keep_idx.size), dtype=np.float64)
        exposure_pct = np.empty(int(keep_idx.size), dtype=np.float64)

        evaluate_no_risk_exact_chunk_two(
            selected=selected,
            indicator_pools=indicator_pools,
            exact_strategy=exact_strategy,
            total_return_pct=total_return_pct,
            max_drawdown_pct=max_drawdown_pct,
            return_over_max_drawdown=return_over_max_drawdown,
            profit_factor=profit_factor,
            trade_count=trade_count,
            sharpe_trades=sharpe_trades,
            win_rate_pct=win_rate_pct,
            avg_trade_ret_pct=avg_trade_ret_pct,
            avg_trade_exec_bars=avg_trade_exec_bars,
            exposure_pct=exposure_pct,
        )

        for local_idx in range(int(keep_idx.size)):
            score = float(total_return_pct[local_idx])
            dema_local_idx = int(selected["ma.dema"][local_idx])
            hma_local_idx = int(selected["ma.hma"][local_idx])
            dema_orig_row = int(indicator_pools["ma.dema"]["row_ids"][dema_local_idx])
            hma_orig_row = int(indicator_pools["ma.hma"]["row_ids"][hma_local_idx])
            proxy_pending = selected_confirm is None
            if proxy_pending:
                confirm_count = 0
                proxy_score = 0.0
            else:
                confirm_count = int(selected_confirm[local_idx])
                proxy_score = float(selected_proxy[local_idx])
            item = {
                "total_return_pct": score,
                "confirm_count": confirm_count,
                "proxy_score": proxy_score,
                "trade_count": int(trade_count[local_idx]),
                "max_drawdown_pct": float(max_drawdown_pct[local_idx]),
                "return_over_max_drawdown": float(return_over_max_drawdown[local_idx]),
                "profit_factor": float(profit_factor[local_idx]),
                "sharpe_trades": float(sharpe_trades[local_idx]),
                "win_rate_pct": float(win_rate_pct[local_idx]),
                "avg_trade_ret_pct": float(avg_trade_ret_pct[local_idx]),
                "avg_trade_exec_bars": float(avg_trade_exec_bars[local_idx]),
                "exposure_pct": float(exposure_pct[local_idx]),
                "ma.dema": indicator_pools["ma.dema"]["metadata"][dema_local_idx],
                "ma.hma": indicator_pools["ma.hma"]["metadata"][hma_local_idx],
                "_dema_local_idx": dema_local_idx,
                "_hma_local_idx": hma_local_idx,
                "_proxy_pending": proxy_pending,
            }
            heap_item = (score, dema_orig_row, hma_orig_row, item)
            if len(heap) < top_k:
                heapq.heappush(heap, heap_item)
            elif heap_item[:3] > heap[0][:3]:
                heapq.heapreplace(heap, heap_item)

    filtered_pool_sizes = {
        indicator_id: int(indicator_pools[indicator_id]["trade_T"].shape[0])
        for indicator_id in required
    }
    if verbose:
        print("filtered pool sizes:", filtered_pool_sizes)
        print("combo chunks processed:", total_combo_chunks)
        print("exact candidates evaluated:", total_exact_candidates)
        print("exact backend:", exact_strategy["name"])
        if self_check is not None:
            print("self-check:", self_check)

    top_results = []
    for _, _, _, item in sorted(heap, key=lambda pair: pair[:3], reverse=True):
        dema_local_idx = item.pop("_dema_local_idx")
        hma_local_idx = item.pop("_hma_local_idx")
        if item.pop("_proxy_pending"):
            confirm_count, proxy_score = proxy_for_two_rows(
                indicator_pools["ma.dema"]["eval_T"][dema_local_idx],
                indicator_pools["ma.hma"]["eval_T"][hma_local_idx],
                signal_returns_15m,
                np.int32(combo_min_confirm),
                fee_penalty_per_confirm,
            )
            item["confirm_count"] = int(confirm_count)
            item["proxy_score"] = float(proxy_score)
        top_results.append(item)
    return {
        "filtered_pool_sizes": filtered_pool_sizes,
        "combo_chunks_processed": total_combo_chunks,
        "exact_candidates_evaluated": total_exact_candidates,
        "indicator_pool_schema": INDICATOR_POOL_SCHEMA_KEYS,
        "self_check": self_check,
        "exact_backend": exact_strategy,
        "exact_engine": exact_strategy["name"],
        "top_results": top_results,
    }


In [ ]:
row_pools = {
    "ma.dema": row_ids_for_sources(indicator_id="ma.dema", source_names=["close", "high", "hlc3"]),
    "ma.hma": row_ids_for_sources(indicator_id="ma.hma", source_names=["close", "high", "hlc3"]),
}

result = search_topk_two_indicator_no_risk(
    row_pools=row_pools,
    indicator_top_frac=PREFILTER_TOP_FRAC,
    min_nonzero=PREFILTER_MIN_NONZERO,
    combo_top_frac=COMBO_PREFILTER_TOP_FRAC,
    combo_min_confirm=COMBO_MIN_CONFIRM,
    combo_chunk_size=COMBO_CHUNK_SIZE,
    top_k=TOP_K_DEFAULT,
    self_check_n=2,
    verbose=False,
)

print("filtered pool sizes:", result["filtered_pool_sizes"])
print("combo chunks processed:", result["combo_chunks_processed"])
print("exact candidates evaluated:", result["exact_candidates_evaluated"])
print("self-check:", result["self_check"])


In [ ]:
result["top_results"][:5]